In [1]:
import io
import requests
from Bio import Align, SeqIO

# Target UniProt IDs for comparison
uniprot_ids = {
    "Human": "P28223",
    "Mouse": "P35363"
}

sequences = {}
custom_headers = {"User-Agent": "Mozilla/5.0"}

# Download FASTA records from UniProt
for species, acc_id in uniprot_ids.items():
    url = f"https://rest.uniprot.org/uniprotkb/{acc_id}.fasta"
    res = requests.get(url, headers=custom_headers)
    if res.status_code == 200:
        record = SeqIO.read(io.StringIO(res.text), "fasta")
        sequences[species] = str(record.seq)

# Global pairwise alignment setup
aligner = Align.PairwiseAligner()
aligner.mode = 'global'

human_seq = sequences["Human"]
mouse_seq = sequences["Mouse"]

align_score = aligner.score(human_seq, mouse_seq)

# Calculate sequence identity percentage
seq_len = max(len(human_seq), len(mouse_seq))
identity_percentage = (align_score / seq_len) * 100

print("Alignment Results (Human vs Mouse 5-HT2A)")
print(f"Score: {align_score}")
print(f"Sequence Identity: {identity_percentage:.2f}%\n")

# Find point mutations
mutations = []
for i, (a, b) in enumerate(zip(human_seq, mouse_seq)):
    if a != b:
        mutations.append((i + 1, a, b))

print(f"Total Mismatches: {len(mutations)}")
print("First 5 mismatches (Pos | Human -> Mouse):")
for pos, h_aa, m_aa in mutations[:5]:
    print(f"  Pos {pos}: {h_aa} -> {m_aa}")

Alignment Results (Human vs Mouse 5-HT2A)
Score: 431.0
Sequence Identity: 91.51%

Total Mismatches: 40
First 5 mismatches (Pos | Human -> Mouse):
  Pos 2: D -> E
  Pos 7: E -> D
  Pos 9: T -> I
  Pos 14: T -> I
  Pos 15: T -> P
